# 第 3 天 · async 与 pydantic

> 📓 **这是第 3 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/3)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q httpx pydantic
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
import asyncio
import time


async def fake_io(name: str, seconds: float) -> str:
    await asyncio.sleep(seconds)          # 模拟慢 IO
    return f"{name} 完成"


async def main() -> None:
    t = time.perf_counter()
    for n in ("A", "B", "C"):
        await fake_io(n, 1)               # 一个一个等：约 3 秒
    print(f"串行耗时 {time.perf_counter() - t:.2f}s")

    t = time.perf_counter()
    results = await asyncio.gather(fake_io("A", 1), fake_io("B", 1), fake_io("C", 1))
    print(f"并发耗时 {time.perf_counter() - t:.2f}s -> {results}")


await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Post(BaseModel):
    id: int
    user_id: int = Field(alias="userId")
    title: str = Field(min_length=1)


print(Post.model_validate({"id": "1", "userId": 7, "title": "hello"}))   # "1" 会被转成 1

try:
    Post.model_validate({"id": "abc", "userId": 7, "title": ""})
except ValidationError as e:
    for err in e.errors():
        print("字段", err["loc"], "→", err["msg"])

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = ["httpx>=0.27", "pydantic>=2.7"]
# ///
"""第 3 天：asyncio + httpx + pydantic（Java 对照版）。运行：uv run day03_async.py"""
from __future__ import annotations

import asyncio
import time

import httpx
from pydantic import BaseModel, Field, ValidationError, field_validator


# ---------- 1. pydantic 模型 ≈ DTO + Bean Validation ----------
class Company(BaseModel):
    name: str
    catch_phrase: str = Field(alias="catchPhrase")     # ≈ @JsonProperty("catchPhrase")


class User(BaseModel):
    id: int
    name: str
    email: str
    company: Company                                    # 嵌套对象自动解析

    @field_validator("email")                           # ≈ 自定义 @Email 校验器
    @classmethod
    def must_have_at(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError("邮箱格式不对")
        return v.lower()


class Post(BaseModel):
    id: int
    user_id: int = Field(alias="userId")
    title: str = Field(min_length=1)                    # ≈ @NotBlank


class Todo(BaseModel):
    id: int
    title: str
    completed: bool


BASE = "https://jsonplaceholder.typicode.com"


# ---------- 2. async 函数 ≈ 返回 CompletableFuture 的方法 ----------
async def fetch_json(client: httpx.AsyncClient, path: str) -> dict | list:
    resp = await client.get(f"{BASE}{path}")            # await ≈ future.join()，但不阻塞线程
    resp.raise_for_status()                             # 4xx/5xx 抛 httpx.HTTPStatusError
    return resp.json()


async def fake_io(name: str, seconds: float) -> str:
    await asyncio.sleep(seconds)                        # 模拟一次慢 IO（不能用 time.sleep！）
    return f"{name} 完成"


async def demo_sequential_vs_gather() -> None:
    print("== 串行 vs 并发 ==")
    t = time.perf_counter()
    for n in ("A", "B", "C"):
        await fake_io(n, 1)                             # 一个一个等：约 3 秒
    print(f"串行耗时 {time.perf_counter() - t:.2f}s")

    t = time.perf_counter()
    results = await asyncio.gather(                     # ≈ CompletableFuture.allOf(...)
        fake_io("A", 1), fake_io("B", 1), fake_io("C", 1)
    )
    print(f"并发耗时 {time.perf_counter() - t:.2f}s -> {results}")


async def demo_real_api() -> None:
    print("\n== 并发请求 3 个接口 + pydantic 解析 ==")
    # async with ≈ try-with-resources，结束时自动关闭连接池
    async with httpx.AsyncClient(timeout=10) as client:
        t = time.perf_counter()
        user_raw, posts_raw, todos_raw = await asyncio.gather(
            fetch_json(client, "/users/1"),
            fetch_json(client, "/posts?userId=1"),
            fetch_json(client, "/todos?userId=1"),
        )
        print(f"3 个请求并发完成，耗时 {time.perf_counter() - t:.2f}s")

    user = User.model_validate(user_raw)                # ≈ objectMapper.readValue + @Valid
    posts = [Post.model_validate(p) for p in posts_raw]
    todos = [Todo.model_validate(x) for x in todos_raw]
    done = sum(x.completed for x in todos)

    print(f"用户：{user.name} <{user.email}>，公司：{user.company.name}")
    print(f"帖子 {len(posts)} 篇，第一篇：{posts[0].title[:30]}...")
    print(f"待办 {len(todos)} 条，已完成 {done} 条")
    print("转回 JSON：", user.model_dump_json(by_alias=True)[:80], "...")


def demo_validation_error() -> None:
    print("\n== 校验失败长什么样 ==")
    bad = {"id": "不是数字", "name": "Tom", "email": "no-at-sign",
           "company": {"name": "X"}}                    # 少了 catchPhrase
    try:
        User.model_validate(bad)
    except ValidationError as e:                        # ≈ MethodArgumentNotValidException
        for err in e.errors():
            print(f"  字段 {err['loc']}: {err['msg']}")


async def demo_error_isolation() -> None:
    print("\n== 某个请求失败时不拖垮其它请求 ==")
    async with httpx.AsyncClient(timeout=10) as client:
        results = await asyncio.gather(
            fetch_json(client, "/users/2"),
            fetch_json(client, "/users/99999"),          # 404
            return_exceptions=True,                      # 异常作为结果返回，而不是直接抛出
        )
    for r in results:
        if isinstance(r, Exception):
            print("  失败：", type(r).__name__)
        else:
            print("  成功：", r["name"])


async def main() -> None:
    await demo_sequential_vs_gather()
    await demo_real_api()
    demo_validation_error()
    await demo_error_isolation()


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await

In [ ]:
from __future__ import annotations

import asyncio
from datetime import datetime

import httpx
from pydantic import BaseModel

REPOS = ["python/cpython", "langchain-ai/langgraph", "spring-projects/spring-boot"]


class Repo(BaseModel):
    full_name: str
    stargazers_count: int
    # TODO: forks_count / language / pushed_at


async def fetch_repo(client: httpx.AsyncClient, name: str) -> Repo:
    resp = await client.get(f"https://api.github.com/repos/{name}")
    resp.raise_for_status()
    return Repo.model_validate(resp.json())


async def main() -> None:
    async with httpx.AsyncClient(timeout=10) as client:
        repos = await asyncio.gather(*(fetch_repo(client, n) for n in REPOS))
    for r in sorted(repos, key=lambda r: r.stargazers_count, reverse=True):
        print(f"{r.full_name:35} ⭐ {r.stargazers_count:>7}")


await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await